In [ ]:
import pandas as pd

# We can also use Snowpark for our analyses!
from snowflake.snowpark.context import get_active_session
from snowflake.ml.modeling.ensemble import RandomForestRegressor
from snowflake.ml.registry import Registry


In [ ]:
session = get_active_session()

In [ ]:
session.query_tag="model-registry-1"

In [ ]:
df=session.table(name="test.diamonds.diamonds")

In [ ]:
df.show()

In [ ]:
train_df, test_df=df.random_split(weights=[0.9,0.1], seed=0)

In [ ]:
train_df.show()

In [ ]:
train_df.columns

In [ ]:
model=RandomForestRegressor(
    input_cols=['DEPTH',"TABLENO","X","Y","Z"],
    label_cols=['PRICE'],
    output_cols=['PREDICTED_PRICE']
)

In [ ]:
model.fit(dataset=train_df)

In [ ]:
pred=model.predict(dataset=test_df)

In [ ]:
pred.select(
    "PRICE",
    "PREDICTED_PRICE"
).show()

In [ ]:
registry=Registry(session=session)

In [ ]:
# Register the model
model_ref=registry.log_model(
    model,
    model_name="RandomForestDiamonds",
    version_name="v2",
    conda_dependencies=["scikit-learn"]
)

In [ ]:
# Register the model in the warehouse as well as container services
model_ref=registry.log_model(
    model,
    model_name="RandomForestDiamondsWH",
    version_name="v3",
    target_platforms=['WAREHOUSE','SNOWPARK_CONTAINER_SERVICES'],
    conda_dependencies=["scikit-learn"]
)

In [ ]:
# Registering another model
# Register the model in the warehouse as well as container services
model_ref=registry.log_model(
    model,
    model_name="RandomForestDiamondsWH",
    version_name="v1",
    target_platforms=['WAREHOUSE','SNOWPARK_CONTAINER_SERVICES'],
    conda_dependencies=["scikit-learn"]
)

In [ ]:
registry.show_models()

In [ ]:
pd.DataFrame(
    data=registry.show_models(m)
)

In [ ]:
LoadModel=registry.get_model(
    model_name="RandomForestDiamondsWH"
)

In [ ]:
LoadModel=LoadModel.version("v1")

In [ ]:
LoadModel

In [ ]:
predictions=LoadModel.run(
    test_df,
    function_name="predict"
)

In [ ]:
predictions.select('PRICE',"PREDICTED_PRICE").show()